In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import load_npz
import joblib

# Загрузка данных
print("Загрузка обработанных данных...")
train_fe = pd.read_parquet('data/processed/train_fe.parquet')
queries_fe = pd.read_parquet('data/processed/queries_fe.parquet')
items_fe = pd.read_parquet('data/processed/items_fe.parquet')

print(f"Данные загружены:")
print(f"  train: {train_fe.shape}")
print(f"  queries: {queries_fe.shape}")
print(f"  items: {items_fe.shape}")

Загрузка обработанных данных...
Данные загружены:
  train: (497673, 31)
  queries: (2452, 17)
  items: (189212, 37)


In [6]:
item_tfidf_matrix = load_npz('data/processed/item_tfidf_matrix.npz')
tfidf_vectorizer = joblib.load('data/processed/tfidf_vectorizer.pkl')

def find_candidates_with_location_filter(query_row, tfidf_vectorizer, item_tfidf_matrix, items_df, top_n=50, location_boost=True):

    query_text = query_row['search_query_clean']
    query_location = query_row.get('search_location_filled', None)
    
    # Базовый TF-IDF поиск
    query_tfidf = tfidf_vectorizer.transform([query_text])
    similarities = cosine_similarity(query_tfidf, item_tfidf_matrix).flatten()
    
    # Если есть локация и включен бустинг
    if query_location and query_location != -1 and location_boost:
        # Находим объявления из той же локации
        same_location_mask = items_df['item_location_filled'] == query_location
        same_location_count = same_location_mask.sum()
        
        if same_location_count > 0:
            # Бустим сходство для объявлений из той же локации
            boost_factor = 2.0  # множитель для буста
            similarities[same_location_mask.values] *= boost_factor
            
            print(f"  Буст для {same_location_count} объявлений из локации {query_location}")
    
    # Сортируем и берем топ-N
    top_indices = np.argsort(similarities)[::-1][:top_n]
    top_item_ids = items_df.iloc[top_indices]['item_id'].tolist()
    
    return top_item_ids

# Тест
print("Тест поиска с фильтрацией по локации...")
test_query_row = queries_fe.iloc[3]
print(f"Запрос: '{test_query_row['search_query_clean']}'")
print(f"Локация: {test_query_row.get('search_location_filled', 'N/A')}")

candidates_loc = find_candidates_with_location_filter(
    test_query_row, 
    tfidf_vectorizer, 
    item_tfidf_matrix, 
    items_fe, 
    top_n=10,
    location_boost=True
)

print(f"\nНайдено {len(candidates_loc)} кандидатов:")
for i, item_id in enumerate(candidates_loc[:5], 1):
    item = items_fe[items_fe['item_id'] == item_id].iloc[0]
    loc_match = "Yes" if item['item_location_filled'] == test_query_row.get('search_location_filled') else "No"
    print(f"  {i}. {loc_match} {item['item_title_raw'][:60]} (локация: {item['item_location_filled']})")

Тест поиска с фильтрацией по локации...
Запрос: 'подъемник 4 х стоечный'
Локация: 662810
  Буст для 1133 объявлений из локации 662810

Найдено 10 кандидатов:
  1. Yes Аренда и услуги ножничного подъёмника (локация: 662810)
  2. Yes Коленчатый подъемник 20 м без залога (локация: 662810)
  3. Yes Аренда ножничного, коленчатого подъемника (локация: 662810)
  4. No Подъемник для гипсокартона аренда (локация: 653341)
  5. No Сдам в аренду (локация: 662420)


In [7]:
from sentence_transformers import SentenceTransformer
import torch

print("Загрузка Sentence Transformer модели\n")

# Проверяем доступность GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Используемое устройство: {device}")

# Загрузка модели multilingual
model_name = 'paraphrase-multilingual-mpnet-base-v2'
print(f"Загрузка модели: {model_name}")

model = SentenceTransformer(model_name, device=device)

print(f"Модель загружена")
print(f"   Размерность эмбеддингов: {model.get_embedding_dimension()}")

Загрузка Sentence Transformer модели

Используемое устройство: cpu
Загрузка модели: paraphrase-multilingual-mpnet-base-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Модель загружена
   Размерность эмбеддингов: 768


In [9]:
print("Создание эмбеддингов для объявлений\n")

item_texts = items_fe['item_text_combined'].fillna('').tolist()

print(f"Создание эмбеддингов для {len(item_texts)} объявлений")

# Создание эмбеддингов батчами
batch_size = 256
item_embeddings = model.encode(
    item_texts, 
    batch_size=batch_size, 
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"\nЭмбеддинги созданы:")
print(f"   Форма: {item_embeddings.shape}")
print(f"   Тип: {item_embeddings.dtype}")

# Сохранение эмбеддингов
np.save('data/processed/item_embeddings.npy', item_embeddings)
print(f"   Сохранено в item_embeddings.npy")

Создание эмбеддингов для объявлений

Создание эмбеддингов для 189212 объявлений


Batches:   0%|          | 0/740 [00:00<?, ?it/s]


Эмбеддинги созданы:
   Форма: (189212, 768)
   Тип: float32
   Сохранено в item_embeddings.npy


In [10]:
from sklearn.metrics.pairwise import cosine_similarity

def find_candidates_embeddings(query_text, model, item_embeddings, items_df, top_n=50):
    # Эмбеддинги запросов
    query_embedding = model.encode([query_text], convert_to_numpy=True)
    
    # Считаем косинусное сходство
    similarities = cosine_similarity(query_embedding, item_embeddings).flatten()
    
    # Находим топ-N
    top_indices = np.argsort(similarities)[::-1][:top_n]
    top_item_ids = items_df.iloc[top_indices]['item_id'].tolist()
    
    return top_item_ids

# Тест
print("Тест семантического поиска...")
test_query = queries_fe.iloc[0]['search_query_clean']
print(f"Запрос: '{test_query}'")

candidates_emb = find_candidates_embeddings(
    test_query, 
    model, 
    item_embeddings, 
    items_fe, 
    top_n=10
)

print(f"\nНайдено {len(candidates_emb)} кандидатов:")
for i, item_id in enumerate(candidates_emb[:5], 1):
    item = items_fe[items_fe['item_id'] == item_id].iloc[0]
    print(f"  {i}. {item['item_title_raw'][:80]}")

Тест семантического поиска...
Запрос: 'перевозки владикавказ тбилиси'

Найдено 10 кандидатов:
  1. Услуги перевозки людей
  2. Трансфер минивэн из Краснодара в Тбилиси и Ереван
  3. Трансфер из Нальчика в Грузию и Армению
  4. Трансфер Владикавказ - Тбилиси, Батуми, Ереван
  5. Услуги Перевозки Машины из Армении Грузии в Россию


In [11]:
item_tfidf_matrix = load_npz('data/processed/item_tfidf_matrix.npz')
tfidf_vectorizer = joblib.load('data/processed/tfidf_vectorizer.pkl')

def find_candidates_ensemble(query_row, tfidf_vectorizer, item_tfidf_matrix, 
                             model, item_embeddings, items_df, 
                             top_n=50, tfidf_weight=0.5, emb_weight=0.5,
                             location_boost=True):

    query_text = query_row['search_query_clean']
    query_location = query_row.get('search_location_filled', None)
    
    # TF-IDF сходство
    query_tfidf = tfidf_vectorizer.transform([query_text])
    tfidf_similarities = cosine_similarity(query_tfidf, item_tfidf_matrix).flatten()
    
    # Семантическое сходство
    query_embedding = model.encode([query_text], convert_to_numpy=True)
    emb_similarities = cosine_similarity(query_embedding, item_embeddings).flatten()
    
    # Нормализуем сходства (приводим к [0, 1])
    tfidf_similarities_norm = (tfidf_similarities - tfidf_similarities.min()) / (tfidf_similarities.max() - tfidf_similarities.min() + 1e-8)
    emb_similarities_norm = (emb_similarities - emb_similarities.min()) / (emb_similarities.max() - emb_similarities.min() + 1e-8)
    
    # Комбинируем
    combined_similarities = tfidf_weight * tfidf_similarities_norm + emb_weight * emb_similarities_norm
    
    # Буст по локации
    if query_location and query_location != -1 and location_boost:
        same_location_mask = items_df['item_location_filled'] == query_location
        if same_location_mask.sum() > 0:
            combined_similarities[same_location_mask.values] *= 1.5
    
    # Топ-N
    top_indices = np.argsort(combined_similarities)[::-1][:top_n]
    top_item_ids = items_df.iloc[top_indices]['item_id'].tolist()
    
    return top_item_ids

# Тест ансамбля
print("Тест ансамбля моделей...")
test_query_row = queries_fe.iloc[0]
print(f"Запрос: '{test_query_row['search_query_clean']}'")

candidates_ensemble = find_candidates_ensemble(
    test_query_row, 
    tfidf_vectorizer, 
    item_tfidf_matrix, 
    model, 
    item_embeddings, 
    items_fe, 
    top_n=10,
    tfidf_weight=0.4,
    emb_weight=0.6,
    location_boost=True
)

print(f"\nНайдено {len(candidates_ensemble)} кандидатов:")
for i, item_id in enumerate(candidates_ensemble[:5], 1):
    item = items_fe[items_fe['item_id'] == item_id].iloc[0]
    print(f"  {i}. {item['item_title_raw'][:80]}")

Тест ансамбля моделей...
Запрос: 'перевозки владикавказ тбилиси'

Найдено 10 кандидатов:
  1. Услуги перевозки людей
  2. Трансфер Владикавказ-Гудаури-Тбилиси
  3. Трансфер Владикавказ Беслан Тбилиси
  4. Трансфер Владикавказ - Тбилиси.Батуми Гудаури
  5. Трансфер в Тбилиси,Гудаури,поездки в Грузию


In [12]:
import numpy as np
import os

os.makedirs('data/processed', exist_ok=True)

np.save('data/processed/item_embeddings.npy', item_embeddings)
print(f"Эмбеддинги сохранены. Размер файла: {os.path.getsize('data/processed/item_embeddings.npy') / 1024**2:.1f} MB")

Эмбеддинги сохранены. Размер файла: 554.3 MB


In [13]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import pickle
from scipy.sparse import load_npz
import joblib

item_embeddings = np.load('data/processed/item_embeddings.npy')
train_fe = pd.read_parquet('data/processed/train_fe.parquet')
queries_fe = pd.read_parquet('data/processed/queries_fe.parquet')
items_fe = pd.read_parquet('data/processed/items_fe.parquet')
item_tfidf_matrix = load_npz('data/processed/item_tfidf_matrix.npz')
tfidf_vectorizer = joblib.load('data/processed/tfidf_vectorizer.pkl')

print("Генерация предсказаний ансамбля\n")

BEST_TFIDF_WEIGHT = 0.4
BEST_EMB_WEIGHT = 0.6
USE_LOCATION_BOOST = True

def find_candidates_ensemble(query_row, tfidf_vectorizer, item_tfidf_matrix, 
                             model, item_embeddings, items_df, 
                             top_n=50, tfidf_weight=0.4, emb_weight=0.6,
                             location_boost=True):
    query_text = query_row['search_query_clean']
    query_location = query_row.get('search_location_filled', None)
    
    # TF-IDF сходство
    query_tfidf = tfidf_vectorizer.transform([query_text])
    tfidf_similarities = cosine_similarity(query_tfidf, item_tfidf_matrix).flatten()
    
    # Семантическое сходство
    query_embedding = model.encode([query_text], convert_to_numpy=True)
    emb_similarities = cosine_similarity(query_embedding, item_embeddings).flatten()
    
    # Нормализация в [0, 1]
    tfidf_similarities_norm = (tfidf_similarities - tfidf_similarities.min()) / (tfidf_similarities.max() - tfidf_similarities.min() + 1e-8)
    emb_similarities_norm = (emb_similarities - emb_similarities.min()) / (emb_similarities.max() - emb_similarities.min() + 1e-8)
    
    # Комбинирование
    combined_similarities = tfidf_weight * tfidf_similarities_norm + emb_weight * emb_similarities_norm
    
    # Буст по локации
    if query_location and query_location != -1 and location_boost:
        same_location_mask = items_df['item_location_filled'] == query_location
        if same_location_mask.sum() > 0:
            combined_similarities[same_location_mask.values] *= 1.5
    
    # Топ-N
    top_indices = np.argsort(combined_similarities)[::-1][:top_n]
    top_item_ids = items_df.iloc[top_indices]['item_id'].tolist()
    
    return top_item_ids


def generate_ensemble_predictions(queries_df, tfidf_vectorizer, item_tfidf_matrix,
                                  model, item_embeddings, items_df,
                                  top_n=50, tfidf_weight=0.4, emb_weight=0.6,
                                  location_boost=True):
    predictions = {}
    total = len(queries_df)
    
    print(f"Генерация предсказаний для {total} запросов...")
    print(f"Параметры: tfidf_weight={tfidf_weight}, emb_weight={emb_weight}, location_boost={location_boost}")
    
    for idx, row in queries_df.iterrows():
        query_id = row['query_id']
        
        candidates = find_candidates_ensemble(
            row, 
            tfidf_vectorizer, 
            item_tfidf_matrix, 
            model, 
            item_embeddings, 
            items_df,
            top_n=top_n,
            tfidf_weight=tfidf_weight,
            emb_weight=emb_weight,
            location_boost=location_boost
        )
        
        predictions[query_id] = candidates
        
        if (idx + 1) % 100 == 0:
            print(f"  Обработано {idx + 1}/{total} ({(idx+1)/total*100:.1f}%)")
    
    print(f"\nСгенерировано предсказаний для {len(predictions)} запросов")
    return predictions


# Запускаем генерацию
predictions_ensemble = generate_ensemble_predictions(
    queries_fe,
    tfidf_vectorizer,
    item_tfidf_matrix,
    model,
    item_embeddings,
    items_fe,
    top_n=50,
    tfidf_weight=BEST_TFIDF_WEIGHT,
    emb_weight=BEST_EMB_WEIGHT,
    location_boost=USE_LOCATION_BOOST
)

# Сохраняем предсказания ансамбля
with open('data/processed/predictions_ensemble.pkl', 'wb') as f:
    pickle.dump(predictions_ensemble, f)
print("Предсказания сохранены в predictions_ensemble.pkl")

Генерация предсказаний ансамбля

Генерация предсказаний для 2452 запросов...
Параметры: tfidf_weight=0.4, emb_weight=0.6, location_boost=True
  Обработано 100/2452 (4.1%)
  Обработано 200/2452 (8.2%)
  Обработано 300/2452 (12.2%)
  Обработано 400/2452 (16.3%)
  Обработано 500/2452 (20.4%)
  Обработано 600/2452 (24.5%)
  Обработано 700/2452 (28.5%)
  Обработано 800/2452 (32.6%)
  Обработано 900/2452 (36.7%)
  Обработано 1000/2452 (40.8%)
  Обработано 1100/2452 (44.9%)
  Обработано 1200/2452 (48.9%)
  Обработано 1300/2452 (53.0%)
  Обработано 1400/2452 (57.1%)
  Обработано 1500/2452 (61.2%)
  Обработано 1600/2452 (65.3%)
  Обработано 1700/2452 (69.3%)
  Обработано 1800/2452 (73.4%)
  Обработано 1900/2452 (77.5%)
  Обработано 2000/2452 (81.6%)
  Обработано 2100/2452 (85.6%)
  Обработано 2200/2452 (89.7%)
  Обработано 2300/2452 (93.8%)
  Обработано 2400/2452 (97.9%)

Сгенерировано предсказаний для 2452 запросов
Предсказания сохранены в predictions_ensemble.pkl
